In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [6]:
from my_preprocessing_classes import CategoricalEncoder, reduce_mem_usage, TimeFeaturesEncoder, GroupAggregationTransformer, DropMissingFeatures

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [7]:
!pip install dagshub
!pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 7.8 MB/s eta 0:00:00
  Attempting uninstall: dacite
    Found existing installation: dacite 1.9.2
    Uninstalling dacite-1.9.2:
      Successfully uninstalled dacite-1.9.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [8]:
import dagshub
import mlflow
dagshub.init(repo_owner='mkekn23', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=09479825-91d3-4c20-a506-da7eec91a434&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e10312c6a4598e0579e40fbf709c22dd6e41220381e4107bf5426b5be0ecc1c3




Accessing as mkekn23

Initialized MLflow to track repo "mkekn23/IEEE-CIS-Fraud-Detection"

Repository mkekn23/IEEE-CIS-Fraud-Detection initialized!

# data loading and reducing ram usage

In [9]:
df_identity= pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
df_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
df = pd.merge(df_transaction, df_identity, on='TransactionID', how='left')

In [10]:
del df_transaction, df_identity

In [11]:
df.shape

(590540, 434)

In [12]:
X = df.drop('isFraud', axis=1)
y = df['isFraud']

X = reduce_mem_usage(X)
y = y.astype('int8') 
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Memory usage of dataframe is 1950.87 MB


/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/kaggle/usr/lib/notebooks/mariamkeknadze/my_preprocessing_classes/my_preprocessing_classes.py:41: RuntimeWarning: overflow e

Memory usage after optimization is: 524.99 MB
Decreased by 73.1%


In [13]:
X_train.shape

(472432, 433)

# training

In [14]:
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score, average_precision_score

def calculate_fraud_metrics(y_true, y_pred, y_probs):
    return {
        "auc_roc": roc_auc_score(y_true, y_probs),
        "pr_auc": average_precision_score(y_true, y_probs), # მნიშვნელოვანია დისბალანსისას
        "recall": recall_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred)
    }

import mlflow

def run_experiment(pipeline, run_name, params=None):
    with mlflow.start_run(run_name=run_name):
        if params:
            mlflow.log_params(params)
        
        pipeline.fit(X_train, y_train)
        
        y_pred = pipeline.predict(X_test)
        y_probs = pipeline.predict_proba(X_test)[:, 1]
        
        metrics = calculate_fraud_metrics(y_test, y_pred, y_probs)
        

        mlflow.log_metrics(metrics)
        
        mlflow.sklearn.log_model(pipeline, "model")
        
        print(f"Run '{run_name}' completed. AUC: {metrics['auc_roc']:.4f}")

In [15]:
from sklearn.pipeline import Pipeline

In [16]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer


dt_pipeline_base = Pipeline([
    ('drop_nan', DropMissingFeatures(threshold=0.9)),
    ('cat_enc', CategoricalEncoder()),            
    ('imputer', SimpleImputer(strategy='median')),
    ('model', DecisionTreeClassifier(
        max_depth=10,              
        class_weight='balanced',    
        random_state=42
    ))
])

In [17]:
X_train_copy = X_train.copy()
y_train_copy=y_train.copy()
dt_pipeline_base.fit(X_train_copy, y_train_copy)

y_pred_prob = dt_pipeline_base.predict_proba(X_test)[:, 1]

In [18]:
mlflow.set_experiment("Decition Tree")

<Experiment: artifact_location='mlflow-artifacts:/26d262ad516b4c35b876a5ead2990778', creation_time=1777804833269, experiment_id='1', last_update_time=1777804833269, lifecycle_stage='active', name='Decition Tree', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [19]:
dt_params = {
    "model_type": "DecisionTree",
    "max_depth": 10,
    "class_weight": "balanced",
    "nan_threshold": 0.9,
    "impute_strategy": "median"
}


run_experiment(dt_pipeline_base, "DT_Baseline", params=dt_params)

🏃 View run DT_Baseline at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/457f6fc2792c4406a37e3acde0bcbe41
🧪 View experiment at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1


KeyboardInterrupt: 

In [20]:
dt_pipeline_enhanced = Pipeline([
    ('time_enc', TimeFeaturesEncoder()),     
    ('aggr', GroupAggregationTransformer()),
    ('drop_nan', DropMissingFeatures(threshold=0.9)),
    ('cat_enc', CategoricalEncoder()),            
    ('imputer', SimpleImputer(strategy='median')),
    ('model', DecisionTreeClassifier(
        max_depth=10,              
        class_weight='balanced',    
        random_state=42
    ))
])

X_train_copy = X_train.copy()
y_train_copy=y_train.copy()
dt_pipeline_enhanced.fit(X_train_copy, y_train_copy)

y_pred_prob = dt_pipeline_enhanced.predict_proba(X_test)[:, 1]

In [21]:
dt_params = {
    "model_type": "DecisionTree",
    "max_depth": 10,
    "class_weight": "balanced",
    "nan_threshold": 0.9,
    "impute_strategy": "median"
}


run_experiment(dt_pipeline_enhanced, "DT_new_features", params=dt_params)

2026/05/03 11:14:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 11:14:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 'DT_new_features' completed. AUC: 0.8701
🏃 View run DT_new_features at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/4d79c4e1c8ec45499d274d2096f0b729
🧪 View experiment at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1


In [23]:
from sklearn.model_selection import RandomizedSearchCV


param_dist = {
    'model__max_depth': [5, 10, 15, 20, None],
    'model__min_samples_split': [2, 5, 10],
    'model__criterion': ['gini', 'entropy'],
    'model__max_features': ['sqrt', 'log2', None]
}

In [24]:
def run_optimization(pipeline, run_name, param_dist):
    with mlflow.start_run(run_name=run_name):
        search = RandomizedSearchCV(
            pipeline, 
            param_distributions=param_dist, 
            n_iter=10, 
            cv=3, 
            scoring='roc_auc', 
            n_jobs=-1, 
            random_state=42
        )
        
        search.fit(X_train, y_train)
        
        # საუკეთესო მოდელის აღება
        best_pipeline = search.best_estimator_
        
        # პარამეტრების დალოგვა
        mlflow.log_params(search.best_params_)
        
        # მეტრიკების დათვლა საუკეთესო მოდელისთვის
        y_pred = best_pipeline.predict(X_test)
        y_probs = best_pipeline.predict_proba(X_test)[:, 1]
        metrics = calculate_fraud_metrics(y_test, y_pred, y_probs)
        
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(best_pipeline, "best_model")
        
        print(f"Optimization '{run_name}' done. Best AUC: {search.best_score_:.4f}")
        return best_pipeline

In [25]:
run_optimization(dt_pipeline_enhanced, "DT_Hyperparameter_Tuning", param_dist)

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


2026/05/03 11:26:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 11:26:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Optimization 'DT_Hyperparameter_Tuning' done. Best AUC: 0.8449
🏃 View run DT_Hyperparameter_Tuning at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/7ff7ccc85dd5425ca0a8da0a4eea1ec4
🧪 View experiment at: https://dagshub.com/mkekn23/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1


Pipeline(steps=[('time_enc', TimeFeaturesEncoder()),
                ('aggr', GroupAggregationTransformer()),
                ('drop_nan', DropMissingFeatures()),
                ('cat_enc', CategoricalEncoder()),
                ('imputer', SimpleImputer(strategy='median')),
                ('model',
                 DecisionTreeClassifier(class_weight='balanced', max_depth=10,
                                        max_features='sqrt',
                                        min_samples_split=5,
                                        random_state=42))])

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detect